# Online Shoppers 数据集 Governance 实验

本 notebook 用于运行第二篇 governance 框架在 Online Shoppers 数据集上的实验。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("当前项目目录：", PROJECT_ROOT)


当前项目目录： e:\yan\组\三支决策\机器学习\BT_TWD_v2


In [2]:
CONFIG_PATH = "configs/online_shoppers.yaml"

# 是否运行消融实验：
# True：运行 no_cp 和 no_progressive 两个消融；False：只运行 full，跳过消融单元。
RUN_ABLATION = False

print("当前配置文件：", CONFIG_PATH)
print("运行消融实验：", RUN_ABLATION)


当前配置文件： configs/online_shoppers.yaml
运行消融实验： False


## 结果摘要工具函数


In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display

output_root = Path("outputs/governance")

def show_governance_summary(mode: str) -> None:
    """Read and display the current dataset summary for one run mode."""
    config_stem = Path(CONFIG_PATH).stem
    print(f"\n===== {mode} results =====")

    dataset_summary = output_root / mode / "dataset_summary.csv"
    fold_summary = output_root / mode / config_stem / "fold_summary.csv"
    sample_records = output_root / mode / config_stem / "sample_records.csv"

    if dataset_summary.exists():
        print("Reading:", dataset_summary)
        summary_df = pd.read_csv(dataset_summary)
        if "dataset_name" in summary_df.columns:
            dataset_row = summary_df[summary_df["dataset_name"].astype(str) == config_stem]
            if dataset_row.empty:
                available = ", ".join(summary_df["dataset_name"].astype(str).tolist())
                print(f"No row for {config_stem}; available datasets: {available}")
            else:
                display(dataset_row.reset_index(drop=True))
        else:
            print("dataset_summary.csv has no dataset_name column; cannot filter by dataset.")
    else:
        print("Missing dataset_summary.csv:", dataset_summary)

    if fold_summary.exists():
        print("Reading:", fold_summary)
        display(pd.read_csv(fold_summary))
    else:
        print("Missing fold_summary.csv:", fold_summary)

    if sample_records.exists():
        print("Sample records:", sample_records)
    else:
        print("Missing sample_records.csv:", sample_records)


## 运行完整 governance 实验


In [4]:
import subprocess

cmd = [
    sys.executable,
    "scripts/run_governance_experiments.py",
    "--config",
    CONFIG_PATH,
]

print("运行命令：", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)

print("标准输出：")
print(result.stdout)

print("错误输出：")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"实验运行失败，返回码：{result.returncode}")

show_governance_summary("full")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/online_shoppers.yaml
标准输出：
【INFO】【2026-05-18 19:48:00】【数据加载】文本表格 E:\yan\组\三支决策\机器学习\BT_TWD_v2\data\shopper\online_shoppers_intention.csv 已读取，样本数=12330，列数=18
【INFO】【2026-05-18 19:48:00】【数据加载】目标列 Revenue 已检测为 0/1 标签，跳过映射逻辑
【INFO】【2026-05-18 19:48:00】【数据集信息】名称=online_shoppers，样本数=12330，目标列=Revenue，正类比例=15.47%
【INFO】【2026-05-18 19:48:00】【预处理】连续特征=10个，类别特征=7个
【INFO】【2026-05-18 19:48:00】【预处理】编码后维度=68
【INFO】【2026-05-18 19:48:00】【governance】online_shoppers 第 1/5 折
【INFO】【2026-05-18 19:48:00】【桶树】已为样本生成桶ID，共 28 个组合
【INFO】【2026-05-18 19:48:01】【桶树】已为样本生成桶ID，共 26 个组合
【INFO】【2026-05-18 19:48:07】【governance】weak bucket resolver：strong=18，weak=24
【INFO】【2026-05-18 19:48:07】【桶树】已为样本生成桶ID，共 25 个组合
【INFO】【2026-05-18 19:48:12】【governance】online_shoppers 第 2/5 折
【INFO】【2026-05-18 19:48:12】【桶树】已为样本生成桶ID，共 27 个组合
【INFO】【2026-05-18 19:48:12】【桶树】已为样本生成桶ID，共 27 个组合
【INFO】【2026-05-18 19:48:18】【governance】weak bucket resolver：str

,dataset_name,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,closure_rate,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,online_shoppers,0.224006,0.837954,0.664831,0.878427,0.223439,0.826005,0.660093,0.881022,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Reading: outputs\governance\full\online_shoppers\fold_summary.csv


,dataset_name,fold_id,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,online_shoppers,1,0.196269,0.858863,0.695847,0.890105,0.202352,0.848870,0.697076,0.894972,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,online_shoppers,2,0.244931,0.819492,0.650407,0.877940,0.239862,0.809247,0.643623,0.878751,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,online_shoppers,3,0.212895,0.847501,0.673267,0.879562,0.217356,0.834084,0.665153,0.880373,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,online_shoppers,4,0.215734,0.845123,0.671082,0.879157,0.215734,0.832207,0.669757,0.884428,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,online_shoppers,5,0.250203,0.818793,0.633554,0.865369,0.241890,0.805616,0.624857,0.866586,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Sample records: outputs\governance\full\online_shoppers\sample_records.csv


## 运行无 CP 消融


In [5]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-cp-validation",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无 CP 消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_cp")
else:
    print("RUN_ABLATION=False，跳过无 CP 消融。")


RUN_ABLATION=False，跳过无 CP 消融。


## 运行无渐进更新消融


In [6]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-progressive-update",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无渐进更新消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_progressive")
else:
    print("RUN_ABLATION=False，跳过无渐进更新消融。")


RUN_ABLATION=False，跳过无渐进更新消融。
